# CA2 – Modelling and Evaluation
**Moustafa Almohamed – X00228076**

AI: Theory to Practice (COMPH3004) – UCI Bank Marketing dataset

## 1. Setup and Data Loading
This notebook continues from CA1. It loads the data, repeats the CA1 preprocessing, and then builds and evaluates the models.

In [1]:
# tools for tables and numbers
import pandas as pd
import numpy as np

# load the bank data (columns are separated by ";")
df = pd.read_csv("data/bank-additional-full.csv", sep=";")
print(df.shape)  # (rows, columns)

(41188, 21)


## 2. Cleaning (same steps as CA1)
The cleaning decisions from CA1 are repeated here so the notebook runs on its own.

In [2]:
# remove the 12 exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# pdays = 999 means "never contacted", so turn it into a yes/no column
df["prev_contacted"] = (df["pdays"] != 999).astype(int)

# drop pdays (replaced), duration (leakage) and default (almost all "no")
df = df.drop(columns=["pdays", "duration", "default"])

# "unknown" becomes empty (NaN) so the pipeline can fill it later
# housing and loan keep "unknown" as its own category
unk_cols = ["job", "marital", "education"]
df[unk_cols] = df[unk_cols].replace("unknown", np.nan)

print(df.shape)

(41176, 19)


## 3. Train/Test Split
The data is split 80/20 with the same settings as CA1. The test set is kept aside until the final evaluation.

In [3]:
from sklearn.model_selection import train_test_split

# X = the inputs, y = the answer as 1 (yes) or 0 (no)
X = df.drop(columns="y")
y = (df["y"] == "yes").astype(int)

# 80% train, 20% test, same yes/no mix in both (same split as CA1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Share of yes in train:", round(y_train.mean(), 3))

Train: (32940, 18)  Test: (8236, 18)
Share of yes in train: 0.113


## 4. Preprocessing Pipeline (from CA1)
The CA1 pipeline is fitted once on the training set and then applied to the test set. It gives 21 ready columns for all models.

In [4]:
from functools import partial
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectPercentile, SequentialFeatureSelector, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

# the column groups
num_cols = ["age", "campaign", "previous", "emp.var.rate", "cons.price.idx",
            "cons.conf.idx", "euribor3m", "nr.employed"]
ordinal_cols = ["education"]
binary_cols = ["contact", "housing", "loan"]
nominal_cols = ["job", "marital", "poutcome", "month", "day_of_week"]
edu_order = ["illiterate", "basic.4y", "basic.6y", "basic.9y",
             "high.school", "professional.course", "university.degree"]

# numeric part: log on campaign -> scale -> PCA (4 components)
def log_campaign(data):
    data = data.copy()
    data["campaign"] = np.log1p(data["campaign"])
    return data

num_pipe = Pipeline([
    ("log", FunctionTransformer(log_campaign)),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=4)),
])

# categorical part: fill empty values with the most common one, then encode
encode = ColumnTransformer([
    ("ord", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OrdinalEncoder(categories=[edu_order]))]), ordinal_cols),
    ("bin", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(drop="if_binary", handle_unknown="ignore"))]), binary_cols),
    ("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(drop="first", handle_unknown="ignore"))]), nominal_cols),
])

# feature selection: mutual information (top 75%), then backward elimination
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sfs = SequentialFeatureSelector(LogisticRegression(max_iter=1000, random_state=42),
                                direction="backward", scoring="roc_auc", tol=-0.0001, cv=cv, n_jobs=-1)
cat_pipe = Pipeline([
    ("encoding", encode),
    ("mi", SelectPercentile(partial(mutual_info_classif, random_state=42), percentile=75)),
    ("be", sfs),
])

# full pipeline: numeric + prev_contacted (kept as it is) + categorical
prep = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("prev", "passthrough", ["prev_contacted"]),
    ("cat", cat_pipe, ordinal_cols + binary_cols + nominal_cols),
], sparse_threshold=0)

# learn from the training set only, then apply the same steps to the test set
X_train_ready = prep.fit_transform(X_train, y_train)
X_test_ready = prep.transform(X_test)
print(X_train_ready.shape, X_test_ready.shape)

(32940, 21) (8236, 21)


**Interpretation:** The CA1 pipeline was fitted once on the training set and then applied to the test set, so the test set was not used to learn anything. The result is 21 columns: 4 principal components from the numeric features, `prev_contacted`, and 16 selected categorical columns. There are 32,940 training rows and 8,236 test rows. All the models in this notebook are trained on these 21 columns.

## 5. Logistic Regression
Logistic Regression is our first model. It is made for yes/no problems, it is fast, and its coefficients are easy to explain. We start with a simple version and then handle the class imbalance.

### 5.1 Simple Logistic Regression
A first version with no fix for the class imbalance, checked with 5-fold cross-validation.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

# a simple Logistic Regression (no fix for the imbalance yet)
lr = LogisticRegression(max_iter=1000, random_state=42)

# 5-fold cross-validation: every training customer is predicted
# by a model that did not see them during training
pred = cross_val_predict(lr, X_train_ready, y_train, cv=cv)

# precision, recall and F1 for "no" and "yes"
print(classification_report(y_train, pred, target_names=["no", "yes"], digits=3))

              precision    recall  f1-score   support

          no      0.907     0.987     0.945     29229
         yes      0.658     0.202     0.310      3711

    accuracy                          0.898     32940
   macro avg      0.783     0.595     0.627     32940
weighted avg      0.879     0.898     0.874     32940



**Interpretation:** The simple model has high accuracy (0.898), but this is close to what a model that always predicts "no" would get (0.887). The recall for "yes" is only 0.202, so it misses about 80% of the customers who subscribe. Its precision for "yes" is 0.658, so when it does predict "yes" it is usually right. The model is too cautious because 89% of the training data is "no". The next step handles this class imbalance.

### 5.2 Logistic Regression with class weights
`class_weight="balanced"` makes mistakes on "yes" customers count more, so the model stops ignoring them.

In [6]:
# same model, but mistakes on "yes" customers now count more
lr_bal = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")

# 5-fold cross-validation again, on the training data only
pred_bal = cross_val_predict(lr_bal, X_train_ready, y_train, cv=cv)

print(classification_report(y_train, pred_bal, target_names=["no", "yes"], digits=3))

              precision    recall  f1-score   support

          no      0.948     0.797     0.866     29229
         yes      0.292     0.658     0.404      3711

    accuracy                          0.782     32940
   macro avg      0.620     0.727     0.635     32940
weighted avg      0.874     0.782     0.814     32940



**Interpretation:** With class weights, the recall for "yes" rose from 0.202 to 0.658, so the model now finds about two thirds of the customers who subscribe. Precision fell from 0.658 to 0.292, because the model predicts "yes" much more often and many of those calls are wasted. F1 improved from 0.310 to 0.404, so the balance between recall and precision is better overall. Accuracy fell from 0.898 to 0.782, which shows that accuracy is misleading here: the more useful model has the lower accuracy. For the bank, a missed customer costs more than a wasted call, so this trade-off is acceptable.

## 6. XGBoost
XGBoost builds many small decision trees one after another, where each new tree tries to fix the mistakes of the trees before it. Unlike Logistic Regression, it can learn complex patterns. We start with the default settings and then handle the class imbalance.

### 6.1 Simple XGBoost
A first version with default settings and no fix for the class imbalance.

In [7]:
from xgboost import XGBClassifier

# a simple XGBoost with default settings (no fix for the imbalance yet)
xgb = XGBClassifier(random_state=42, n_jobs=-1)

# 5-fold cross-validation, on the training data only
pred_xgb = cross_val_predict(xgb, X_train_ready, y_train, cv=cv)

print(classification_report(y_train, pred_xgb, target_names=["no", "yes"], digits=3))

              precision    recall  f1-score   support

          no      0.911     0.976     0.942     29229
         yes      0.569     0.253     0.350      3711

    accuracy                          0.894     32940
   macro avg      0.740     0.614     0.646     32940
weighted avg      0.873     0.894     0.876     32940



**Interpretation:** With default settings, XGBoost has a recall of 0.253 for "yes" and an F1 of 0.350. This is slightly better than the simple Logistic Regression (recall 0.202, F1 0.310), but it still misses about three quarters of the customers who subscribe. Accuracy is high (0.894) only because most customers say "no". Like Logistic Regression, the model needs the class imbalance to be handled.

### 6.2 XGBoost with the imbalance fix
`scale_pos_weight` makes mistakes on "yes" customers count more. It is set to the number of "no" customers divided by the number of "yes" customers.

In [8]:
# how many "no" customers there are for each "yes" customer
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(ratio, 2))

# same model, but mistakes on "yes" customers now count more
xgb_bal = XGBClassifier(scale_pos_weight=ratio, random_state=42, n_jobs=-1)

# 5-fold cross-validation again, on the training data only
pred_xgb_bal = cross_val_predict(xgb_bal, X_train_ready, y_train, cv=cv)

print(classification_report(y_train, pred_xgb_bal, target_names=["no", "yes"], digits=3))

scale_pos_weight: 7.88
              precision    recall  f1-score   support

          no      0.938     0.855     0.895     29229
         yes      0.328     0.558     0.413      3711

    accuracy                          0.821     32940
   macro avg      0.633     0.706     0.654     32940
weighted avg      0.870     0.821     0.840     32940



**Interpretation:** With `scale_pos_weight` = 7.88, the recall for "yes" rose from 0.253 to 0.558 and F1 rose from 0.350 to 0.413. Precision fell from 0.569 to 0.328. Compared with Logistic Regression with class weights (precision 0.292, recall 0.658, F1 0.404), XGBoost has a slightly higher F1 and precision but a lower recall. The two models are very close, so XGBoost with default settings does not clearly beat the simpler model. Hyperparameter tuning is the next step for both.

## 7. Hyperparameter Tuning
Each model is tuned with grid search and 5-fold stratified cross-validation on the training data only. The setting with the highest F1 for "yes" is kept. The test set is not used.

### 7.1 Logistic Regression (alpha)
We use `SGDClassifier(loss="log_loss")`, which is Logistic Regression with an `alpha` setting. A bigger alpha means stronger regularisation and a simpler model.

In [10]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV

# Logistic Regression with an alpha setting, still with class weights
sgd_lr = SGDClassifier(loss="log_loss", class_weight="balanced", max_iter=1000, random_state=42)

# the alpha values to try (small = complex, big = simple)
param_grid = {"alpha": [0.00001, 0.0001, 0.001, 0.01, 0.1]}

# try each alpha with 5-fold cross-validation, judged by F1 for "yes"
grid_lr = GridSearchCV(sgd_lr, param_grid, scoring="f1", cv=cv, return_train_score=True, n_jobs=-1)
grid_lr.fit(X_train_ready, y_train)

# training score vs cross-validation score for each alpha
results_lr = pd.DataFrame(grid_lr.cv_results_)[["param_alpha", "mean_train_score", "mean_test_score"]]
results_lr.columns = ["alpha", "train F1", "CV F1"]
print(results_lr.round({"train F1": 3, "CV F1": 3}))
print("Best alpha:", grid_lr.best_params_["alpha"], "| Best CV F1:", round(grid_lr.best_score_, 3))

     alpha  train F1  CV F1
0  0.00001     0.362  0.368
1  0.00010     0.377  0.373
2  0.00100     0.393  0.391
3  0.01000     0.371  0.370
4  0.10000     0.359  0.359
Best alpha: 0.001 | Best CV F1: 0.391


#### Logistic Regression (alpha) with all columns scaled
`SGDClassifier` is sensitive to feature scale, and some columns (for example `education`, 0 to 6) are not scaled. Here a StandardScaler is added before the model, inside a pipeline, so it is fitted within each cross-validation fold.

In [11]:
from sklearn.pipeline import make_pipeline

# scale all 21 columns, then Logistic Regression with alpha
sgd_scaled = make_pipeline(StandardScaler(),
                           SGDClassifier(loss="log_loss", class_weight="balanced", max_iter=1000, random_state=42))

# same alpha values as before (the name now includes the step name)
param_grid_s = {"sgdclassifier__alpha": [0.00001, 0.0001, 0.001, 0.01, 0.1]}

grid_lr_s = GridSearchCV(sgd_scaled, param_grid_s, scoring="f1", cv=cv, return_train_score=True, n_jobs=-1)
grid_lr_s.fit(X_train_ready, y_train)

results_lr_s = pd.DataFrame(grid_lr_s.cv_results_)[["param_sgdclassifier__alpha", "mean_train_score", "mean_test_score"]]
results_lr_s.columns = ["alpha", "train F1", "CV F1"]
print(results_lr_s.round({"train F1": 3, "CV F1": 3}))
print("Best alpha:", grid_lr_s.best_params_["sgdclassifier__alpha"], "| Best CV F1:", round(grid_lr_s.best_score_, 3))

     alpha  train F1  CV F1
0  0.00001     0.357  0.353
1  0.00010     0.385  0.387
2  0.00100     0.402  0.403
3  0.01000     0.404  0.402
4  0.10000     0.402  0.402
Best alpha: 0.001 | Best CV F1: 0.403


**Interpretation:** Without scaling, `SGDClassifier` reached a best CV F1 of 0.391. After scaling all columns inside the pipeline, the best CV F1 rose to 0.403 with alpha = 0.001, which matches the earlier Logistic Regression with class weights (0.404). F1 is almost flat for alpha between 0.001 and 0.1, so the model is not very sensitive to regularisation once alpha is large enough, while a very small alpha (0.00001) performs worse. Training and cross-validation F1 are almost the same for every value, so the model does not overfit. We keep the scaled model with alpha = 0.001.

### 7.2 XGBoost
We tune the number of trees (`n_estimators`), the tree depth (`max_depth`) and the learning rate (`learning_rate`). `scale_pos_weight` stays at 7.88.

In [12]:
# XGBoost with the imbalance fix
xgb_tune = XGBClassifier(scale_pos_weight=ratio, random_state=42, n_jobs=-1)

# the values to try: 2 x 3 x 2 = 12 combinations
param_grid_xgb = {"n_estimators": [100, 300],
                  "max_depth": [3, 5, 7],
                  "learning_rate": [0.05, 0.1]}

grid_xgb = GridSearchCV(xgb_tune, param_grid_xgb, scoring="f1", cv=cv, return_train_score=True)
grid_xgb.fit(X_train_ready, y_train)

# results, best first
results_xgb = pd.DataFrame(grid_xgb.cv_results_)[["param_n_estimators", "param_max_depth", "param_learning_rate",
                                                  "mean_train_score", "mean_test_score"]]
results_xgb.columns = ["trees", "depth", "learning rate", "train F1", "CV F1"]
print(results_xgb.sort_values("CV F1", ascending=False).round({"train F1": 3, "CV F1": 3}))
print("Best:", grid_xgb.best_params_, "| Best CV F1:", round(grid_xgb.best_score_, 3))

    trees  depth  learning rate  train F1  CV F1
4     100      7           0.05     0.500  0.448
10    100      7           0.10     0.554  0.446
3     300      5           0.05     0.502  0.446
8     100      5           0.10     0.482  0.443
7     300      3           0.10     0.470  0.441
5     300      7           0.05     0.591  0.440
1     300      3           0.05     0.451  0.437
9     300      5           0.10     0.553  0.434
2     100      5           0.05     0.452  0.434
6     100      3           0.10     0.442  0.432
11    300      7           0.10     0.673  0.417
0     100      3           0.05     0.411  0.405
Best: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 100} | Best CV F1: 0.448


**Interpretation:** The best setting was 100 trees, depth 7 and learning rate 0.05, with a CV F1 of 0.448. This is higher than XGBoost with default settings (0.413) and higher than tuned Logistic Regression (0.403). The top three settings are very close (0.446 to 0.448). The gap between training and CV F1 is small for the best setting (0.500 vs 0.448), but large for the most complex one (300 trees, depth 7, learning rate 0.1: 0.673 vs 0.417), which shows clear overfitting. A smaller learning rate helped the deeper trees learn without memorising the training data.